# 🛒 LLM-Powered Product Data Normalization Pipeline

**Author:** LLM Product Normalizer Project  
**Stack:** Python · Ollama · llama3.2 · Pydantic · Pandas · FastAPI · Streamlit  
**Mode:** 100% Offline — No cloud API required

---

## Overview

E-commerce scrapers generate inconsistent, noisy product descriptions. This notebook walks through the **full pipeline** that converts raw, messy text into clean, structured CSV data using a locally-running LLM.

### What you'll learn:
1. How to preprocess and clean dirty text data
2. How to engineer prompts for structured JSON extraction
3. How to integrate Ollama for local LLM inference
4. How to validate LLM output with Pydantic
5. How to post-process and export clean data
6. How to visualize pipeline results


## 🏗️ Architecture Diagram

```
messy_products.csv
       │
       ▼
┌─────────────┐    ┌──────────────────┐    ┌──────────────────┐
│ Preprocessor│───▶│  LLM Parser      │───▶│ Pydantic         │
│             │    │  (Ollama/llama3.2│    │ Validator        │
│ • HTML strip│    │                  │    │                  │
│ • Unicode   │    │  Prompt Builder  │    │ • Schema check   │
│ • Dedup     │    │  Retry (x3)      │    │ • Confidence     │
│ • Whitespace│    │  JSON Extraction │    │ • Error routing  │
└─────────────┘    └──────────────────┘    └────────┬─────────┘
                                                     │
                             ┌───────────────────────┤
                             │                       │
                   ┌─────────▼──────┐     ┌──────────▼──────┐
                   │ Post-Processor │     │ Error Collector  │
                   │ • Brand norm.  │     │                  │
                   │ • Category     │     │  errors.csv      │
                   │ • Availability │     └─────────────────-┘
                   └─────────┬──────┘
                             │
                   ┌─────────▼──────┐
                   │  CSV Exporter  │
                   │                │
                   │ clean_products │
                   │     .csv       │
                   └────────────────┘
```


## 1. Imports & Setup

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python version: {sys.version}")

In [ ]:
# Core imports
import json
import re
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Project modules
from src.config import settings
from src.schemas import ProductSchema, RawProduct, NormalizedProduct, ErrorRecord
from src.preprocess import (
    load_raw_csv, preprocess_dataframe, dataframe_to_raw_products,
    clean_description, remove_html, normalize_whitespace
)
from src.utils import safe_json_parse, format_price, chunk_list
from src.postprocess import normalize_brand, normalize_category, normalize_availability
from src.validator import validate_product
from src.exporter import products_to_dataframe, errors_to_dataframe

# Plotting style
plt.style.use('dark_background')
PALETTE = ['#58a6ff', '#3fb950', '#f85149', '#d2a8ff', '#ffa657', '#79c0ff']

print("✅ All imports successful")
print(f"   Ollama model  : {settings.ollama_model}")
print(f"   Ollama host   : {settings.ollama_host}")
print(f"   Input path    : {settings.input_path}")
print(f"   Output path   : {settings.output_path}")

## 2. Load & Explore the Dataset

In [ ]:
# Load raw CSV
raw_df = load_raw_csv(settings.input_path)
print(f"Shape         : {raw_df.shape}")
print(f"Columns       : {list(raw_df.columns)}")
print(f"Missing values: {raw_df['raw_description'].isna().sum()}")
print(f"Duplicates    : {raw_df['raw_description'].duplicated().sum()}")
raw_df.head(3)

In [ ]:
# Examine a few raw descriptions
for i in range(3):
    print(f"\n{'='*60}")
    print(f"Record {i+1}:")
    print(raw_df['raw_description'].iloc[i])

In [ ]:
# Description length distribution
raw_df['desc_length'] = raw_df['raw_description'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor('#0d1117')

# Histogram
axes[0].hist(raw_df['desc_length'], bins=40, color=PALETTE[0], alpha=0.8, edgecolor='none')
axes[0].set_title('Description Length Distribution', color='white', fontsize=13)
axes[0].set_xlabel('Characters', color='#8b949e')
axes[0].set_ylabel('Count', color='#8b949e')
axes[0].tick_params(colors='#8b949e')
for spine in axes[0].spines.values():
    spine.set_edgecolor('#30363d')

# Stats table
stats = raw_df['desc_length'].describe().round(1)
axes[1].axis('off')
table_data = [[k, str(v)] for k, v in stats.items()]
table = axes[1].table(
    cellText=table_data,
    colLabels=['Statistic', 'Value'],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(True)
table.set_fontsize(11)
for (r, c), cell in table.get_celld().items():
    cell.set_facecolor('#161b22' if r > 0 else '#21262d')
    cell.set_text_props(color='white')
    cell.set_edgecolor('#30363d')
axes[1].set_title('Length Statistics', color='white', fontsize=13)

plt.tight_layout()
plt.savefig('description_length.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"Mean description length: {raw_df['desc_length'].mean():.0f} characters")

## 3. Preprocessing Pipeline

The preprocessor applies a sequence of cleaning steps:
1. **Fix encoding** — repair mojibake artifacts
2. **Remove HTML** — strip tags and unescape entities
3. **Normalize Unicode** — NFC normalization
4. **Remove control characters** — invisible characters that confuse the LLM
5. **Normalize bullets** — replace typographic symbols
6. **Normalize punctuation** — curly quotes → straight, em-dash → hyphen
7. **Normalize whitespace** — collapse spaces, limit newlines

In [ ]:
# Demonstrate each preprocessing step
messy_example = """  <b>APPLE</b> iPhone&nbsp;15 Pro\t\t\t128GB\n\n\n
₹79,999 — Special Price\nOnly 2 Left! • Free Charger Included
Flat Rs.5000 OFF with HDFC Card  """

print("BEFORE preprocessing:")
print(repr(messy_example[:120]))
print()

cleaned = clean_description(messy_example)
print("AFTER preprocessing:")
print(repr(cleaned))
print()
print(f"Length: {len(messy_example)} → {len(cleaned)} characters ({len(messy_example)-len(cleaned)} removed)")

In [ ]:
# Run full preprocessing on the dataset
clean_df = preprocess_dataframe(raw_df.copy())

print(f"Records before preprocessing : {len(raw_df)}")
print(f"Records after preprocessing  : {len(clean_df)}")
print(f"Records removed (dup/empty)  : {len(raw_df) - len(clean_df)}")
clean_df.head(3)

In [ ]:
# Convert to RawProduct objects
raw_products = dataframe_to_raw_products(clean_df)
print(f"Total RawProduct objects: {len(raw_products)}")
print(f"\nSample RawProduct:")
print(f"  ID: {raw_products[0].id}")
print(f"  Description: {raw_products[0].raw_description[:80]}...")

## 4. Prompt Engineering

The prompt template is the **most critical component** of the pipeline. It instructs the LLM to:
- Return **only** a JSON object (no markdown, no explanations)
- Follow a **strict schema** with all required fields
- Handle edge cases (missing price → `null`, unknown brand → best guess)
- Self-report **confidence** for downstream filtering

In [ ]:
from src.llm_parser import PromptBuilder

builder = PromptBuilder()

# Show the raw template
print("📄 PROMPT TEMPLATE:")
print("=" * 60)
print(builder.template)
print("=" * 60)

In [ ]:
# Show a built prompt with a real description
sample_desc = raw_products[0].raw_description
built_prompt = builder.build(sample_desc)

print("📤 FULL PROMPT SENT TO LLM:")
print("=" * 60)
print(built_prompt)
print("=" * 60)
print(f"\nPrompt length: {len(built_prompt)} characters")

## 5. LLM Integration (Ollama)

We use `OllamaClient` to send prompts to the locally-running llama3.2 model. The `LLMParser` adds:
- **Retry logic** (3 attempts with exponential back-off)
- **JSON extraction** from noisy LLM output (handles markdown fences)
- **Structured logging** for every attempt

In [ ]:
from src.llm_parser import OllamaClient, LLMParser

# Check Ollama availability
client = OllamaClient()
is_available = client.is_available()
print(f"Ollama available: {is_available}")
print(f"Model: {settings.ollama_model}")

if not is_available:
    print("\n⚠️  Ollama not running. Start it with: ollama serve")
    print("    Then pull the model: ollama pull llama3.2")

In [ ]:
import time

# Process a single product to demonstrate the pipeline
if is_available:
    parser = LLMParser()
    sample = raw_products[0]
    
    print(f"📥 INPUT:")
    print(sample.raw_description)
    print()
    
    start = time.perf_counter()
    llm_data, retries, raw_output = parser.parse(sample.raw_description, sample.id)
    elapsed = time.perf_counter() - start
    
    print(f"📤 LLM OUTPUT (in {elapsed:.2f}s, retries={retries}):")
    print(json.dumps(llm_data, indent=2, ensure_ascii=False) if llm_data else "⚠️ Parse failed")
else:
    print("⚠️ Ollama not available — showing example output")
    llm_data = {
        "product_name": "Apple iPhone 15 128GB",
        "brand": "Apple",
        "category": "Mobile",
        "price": 79999,
        "currency": "INR",
        "offer": "Flat Rs.5000 OFF with HDFC Card",
        "availability": "Limited Stock",
        "delivery": "Ships Tomorrow",
        "seller": "Appario Retail",
        "confidence": 0.95
    }
    retries = 0
    print("Example LLM output:")
    print(json.dumps(llm_data, indent=2))

### safe_json_parse — Handling Noisy LLM Output

LLMs sometimes wrap JSON in markdown code fences despite our instructions. `safe_json_parse` handles three common failure modes:

In [ ]:
from src.utils import safe_json_parse

# Test all three extraction strategies
test_cases = [
    # Case 1: Clean JSON
    ('{"brand": "Apple", "price": 79999}', "Clean JSON"),
    
    # Case 2: Markdown fence
    ('```json\n{"brand": "Samsung"}\n```', "Markdown fence"),
    
    # Case 3: JSON embedded in text
    ('Here is the data: {"brand": "OnePlus", "price": 29999}. Hope this helps!', "Embedded in text"),
    
    # Case 4: Garbage
    ("Sorry, I cannot parse that product.", "Garbage (should return None)"),
]

print("safe_json_parse Test Results:")
print("-" * 50)
for text, label in test_cases:
    result = safe_json_parse(text)
    status = "✅" if result else "❌"
    print(f"{status} {label:<30} → {result}")

## 6. Pydantic Validation

After the LLM returns data, Pydantic validation enforces:
- **Required fields** (product_name, brand, confidence)
- **Type coercion** ("₹79,999" → 79999)
- **Range constraints** (confidence: 0.0–1.0, price: ≥0)
- **Business rules** (confidence threshold filtering)

In [ ]:
from src.schemas import ProductSchema
from pydantic import ValidationError

# Test 1: Valid data
print("Test 1: Valid data")
try:
    p = ProductSchema(
        product_name="Apple iPhone 15 128GB",
        brand="Apple",
        category="Mobile",
        price="79,999",   # string → should be coerced to int
        currency="rs",    # → should normalize to INR
        availability="In Stock",
        confidence=1.5,   # > 1 → should clamp to 1.0
    )
    print(f"  ✅ Passed — price={p.price}, currency={p.currency}, confidence={p.confidence}")
except ValidationError as e:
    print(f"  ❌ Failed: {e}")

# Test 2: Missing required field
print("\nTest 2: Missing product_name")
try:
    p = ProductSchema(
        product_name="",
        brand="Apple",
        category="Mobile",
        confidence=0.9,
        availability="In Stock",
    )
    print(f"  ✅ Passed")
except ValidationError as e:
    errors = e.errors(include_url=False)
    print(f"  ❌ Failed as expected: {errors[0]['msg']}")

# Test 3: Negative price
print("\nTest 3: Negative price")
try:
    p = ProductSchema(
        product_name="Test", brand="Brand", category="Mobile",
        price=-500, confidence=0.8, availability="In Stock"
    )
    print(f"  ✅ Passed")
except ValidationError as e:
    errors = e.errors(include_url=False)
    print(f"  ❌ Failed as expected: {errors[0]['msg']}")

In [ ]:
# Validate the LLM output from earlier
from src.validator import validate_product
from src.schemas import RawProduct, NormalizedProduct, ErrorRecord

raw = raw_products[0]
result = validate_product(raw, llm_data, retry_count=retries)

if isinstance(result, NormalizedProduct):
    print("✅ Validation PASSED")
    print(f"   product_name : {result.product_name}")
    print(f"   brand        : {result.brand}")
    print(f"   category     : {result.category}")
    print(f"   price        : {format_price(result.price, result.currency)}")
    print(f"   availability : {result.availability}")
    print(f"   confidence   : {result.confidence:.2f}")
elif isinstance(result, ErrorRecord):
    print(f"❌ Validation FAILED: {result.error_type}")
    print(f"   Detail: {result.error_detail}")

## 7. Post-Processing — Normalization Maps

After validation, deterministic rules normalize remaining inconsistencies:

In [ ]:
from src.postprocess import normalize_brand, normalize_category, normalize_availability

# Brand normalization examples
brand_tests = [
    ("apple", "Apple"),
    ("SAMSUNG", "Samsung"),
    ("oneplus", "OnePlus"),
    ("mi", "Xiaomi"),
    ("boat", "boAt"),
    ("playstation", "Sony"),
]

print("Brand Normalization:")
for raw_brand, expected in brand_tests:
    result = normalize_brand(raw_brand)
    status = "✅" if result == expected else "⚠️"
    print(f"  {status} '{raw_brand}' → '{result}'")

print()

# Category normalization
cat_tests = [
    ("smartphone", "Mobile"),
    ("earbuds", "Headphones"),
    ("television", "TV"),
    ("sneakers", "Shoes"),
    ("pressure cooker", "Kitchen Appliance"),
]

print("Category Normalization:")
for raw_cat, expected in cat_tests:
    result = normalize_category(raw_cat)
    status = "✅" if result == expected else "⚠️"
    print(f"  {status} '{raw_cat}' → '{result}'")

print()

# Availability normalization
avail_tests = [
    ("available", "In Stock"),
    ("sold out", "Out of Stock"),
    ("pre-order", "Coming Soon"),
    ("only 2 left", "Limited Stock"),
]

print("Availability Normalization:")
for raw_avail, expected in avail_tests:
    result = normalize_availability(raw_avail)
    status = "✅" if result == expected else "⚠️"
    print(f"  {status} '{raw_avail}' → '{result}'")

## 8. Run Full Pipeline on Sample

Here we run the complete pipeline end-to-end on a small sample of 10 records to demonstrate the flow.

In [ ]:
from tqdm.notebook import tqdm
from src.llm_parser import LLMParser, OllamaClient
from src.postprocess import postprocess_product

SAMPLE_SIZE = 10  # Change to process more records

successes = []
failures = []
times = []

if is_available:
    parser = LLMParser()
    sample_products = raw_products[:SAMPLE_SIZE]
    
    print(f"Processing {SAMPLE_SIZE} products...\n")
    
    for raw in tqdm(sample_products, desc="Normalizing", unit="product"):
        t0 = time.perf_counter()
        llm_data, retry_count, llm_raw = parser.parse(raw.raw_description, raw.id)
        elapsed = time.perf_counter() - t0
        times.append(elapsed)
        
        if llm_data is None:
            failures.append(ErrorRecord(
                id=raw.id,
                raw_description=raw.raw_description,
                error_type="json_parse_error",
                error_detail="LLM returned no valid JSON",
                retry_count=retry_count,
                llm_raw_output=llm_raw,
            ))
        else:
            result = validate_product(raw, llm_data, retry_count, llm_raw)
            if isinstance(result, NormalizedProduct):
                successes.append(postprocess_product(result))
            else:
                failures.append(result)
    
    print(f"\n{'='*40}")
    print(f"Total processed   : {SAMPLE_SIZE}")
    print(f"Successful        : {len(successes)}")
    print(f"Failed            : {len(failures)}")
    print(f"Success rate      : {len(successes)/SAMPLE_SIZE*100:.1f}%")
    print(f"Avg time/record   : {sum(times)/len(times):.2f}s")
else:
    print("⚠️ Ollama not available — skipping live pipeline run")
    print("Run 'ollama serve' and 'ollama pull llama3.2' first, then re-run this cell.")

## 9. Export to CSV

In [ ]:
from src.exporter import export_results, products_to_dataframe, errors_to_dataframe

if successes or failures:
    paths = export_results(successes, failures)
    print(f"✅ Clean CSV   → {paths['clean']}")
    print(f"✅ Errors CSV  → {paths['errors']}")
    
    # Preview the output
    df_clean = products_to_dataframe(successes)
    print(f"\nClean products preview ({len(df_clean)} rows):")
    display(df_clean[['product_name', 'brand', 'category', 'price', 'currency', 'availability', 'confidence']].head())
else:
    print("No records to export. Run the pipeline first (requires Ollama).")
    
    # Show example clean CSV if it exists
    if settings.output_path.exists():
        df_clean = pd.read_csv(settings.output_path)
        print(f"\nExisting clean_products.csv ({len(df_clean)} rows):")
        display(df_clean.head())

## 10. Visualization

If the full pipeline has been run via `python -m src.main`, load the complete results for visualization:

In [ ]:
# Load full results if available
if settings.output_path.exists():
    df_viz = pd.read_csv(settings.output_path)
    print(f"Loaded {len(df_viz)} records from {settings.output_path.name}")
elif successes:
    df_viz = products_to_dataframe(successes)
    print(f"Using {len(df_viz)} in-memory records")
else:
    # Create demo data for visualization
    import random
    random.seed(42)
    categories = ['Mobile','Laptop','Headphones','TV','Shoes','Fashion','Kitchen Appliance','Gaming','Book','Furniture']
    brands = ['Apple','Samsung','OnePlus','Sony','Nike','Adidas','Philips','Razer','Penguin','IKEA']
    df_viz = pd.DataFrame({
        'category': random.choices(categories, k=200),
        'brand': random.choices(brands, k=200),
        'price': [random.randint(500, 150000) for _ in range(200)],
        'availability': random.choices(['In Stock','Out of Stock','Limited Stock','Coming Soon'], weights=[60,15,15,10], k=200),
        'confidence': [round(random.uniform(0.55, 0.99), 2) for _ in range(200)],
        'currency': ['INR'] * 200
    })
    print("Using demo data for visualization (run full pipeline for real data)")

In [ ]:
# ── Figure 1: Category Distribution ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('LLM Product Normalizer — Pipeline Analytics', 
             color='white', fontsize=16, fontweight='bold', y=0.98)

ax_style = dict(facecolor='#161b22')

# Plot 1: Category Distribution
ax = axes[0, 0]
ax.set_facecolor(ax_style['facecolor'])
cat_counts = df_viz['category'].value_counts()
bars = ax.barh(cat_counts.index, cat_counts.values, color=PALETTE[:len(cat_counts)], alpha=0.9)
ax.set_title('Category Distribution', color='white', fontsize=12, fontweight='bold')
ax.set_xlabel('Count', color='#8b949e')
ax.tick_params(colors='#8b949e', labelsize=9)
for spine in ax.spines.values(): spine.set_edgecolor('#30363d')
for bar, count in zip(bars, cat_counts.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(count), va='center', color='#8b949e', fontsize=8)

# Plot 2: Price Distribution
ax = axes[0, 1]
ax.set_facecolor(ax_style['facecolor'])
prices = pd.to_numeric(df_viz['price'], errors='coerce').dropna()
ax.hist(prices, bins=30, color=PALETTE[0], alpha=0.85, edgecolor='#0d1117', linewidth=0.5)
ax.set_title('Price Distribution (₹)', color='white', fontsize=12, fontweight='bold')
ax.set_xlabel('Price (₹)', color='#8b949e')
ax.set_ylabel('Count', color='#8b949e')
ax.tick_params(colors='#8b949e')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
for spine in ax.spines.values(): spine.set_edgecolor('#30363d')

# Plot 3: Availability Pie
ax = axes[0, 2]
ax.set_facecolor(ax_style['facecolor'])
avail_counts = df_viz['availability'].value_counts()
pie_colors = ['#3fb950', '#f85149', '#d2a8ff', '#ffa657', '#58a6ff']
wedges, texts, autotexts = ax.pie(
    avail_counts.values, 
    labels=avail_counts.index,
    autopct='%1.1f%%',
    colors=pie_colors[:len(avail_counts)],
    startangle=90,
    wedgeprops={'edgecolor': '#0d1117', 'linewidth': 2}
)
for text in texts: text.set_color('#c9d1d9'); text.set_fontsize(9)
for at in autotexts: at.set_color('white'); at.set_fontweight('bold'); at.set_fontsize(8)
ax.set_title('Availability Distribution', color='white', fontsize=12, fontweight='bold')

# Plot 4: Confidence Distribution
ax = axes[1, 0]
ax.set_facecolor(ax_style['facecolor'])
conf_vals = pd.to_numeric(df_viz['confidence'], errors='coerce').dropna()
ax.hist(conf_vals, bins=20, color=PALETTE[3], alpha=0.85, edgecolor='#0d1117', linewidth=0.5)
ax.axvline(conf_vals.mean(), color='#f85149', linestyle='--', linewidth=1.5, label=f'Mean: {conf_vals.mean():.2f}')
ax.axvline(settings.confidence_threshold, color=PALETTE[1], linestyle=':', linewidth=1.5, label=f'Threshold: {settings.confidence_threshold}')
ax.set_title('LLM Confidence Distribution', color='white', fontsize=12, fontweight='bold')
ax.set_xlabel('Confidence Score', color='#8b949e')
ax.set_ylabel('Count', color='#8b949e')
ax.tick_params(colors='#8b949e')
ax.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='white', fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor('#30363d')

# Plot 5: Top Brands
ax = axes[1, 1]
ax.set_facecolor(ax_style['facecolor'])
top_brands = df_viz['brand'].value_counts().head(10)
bar_colors = [PALETTE[i % len(PALETTE)] for i in range(len(top_brands))]
bars = ax.barh(top_brands.index, top_brands.values, color=bar_colors, alpha=0.9)
ax.set_title('Top 10 Brands', color='white', fontsize=12, fontweight='bold')
ax.set_xlabel('Count', color='#8b949e')
ax.tick_params(colors='#8b949e', labelsize=9)
ax.invert_yaxis()
for spine in ax.spines.values(): spine.set_edgecolor('#30363d')

# Plot 6: Price by Category (Box)
ax = axes[1, 2]
ax.set_facecolor(ax_style['facecolor'])
df_price_cat = df_viz[['category', 'price']].copy()
df_price_cat['price'] = pd.to_numeric(df_price_cat['price'], errors='coerce')
df_price_cat = df_price_cat.dropna(subset=['price'])

cat_order = df_viz['category'].value_counts().index[:8]
bp_data = [df_price_cat[df_price_cat['category']==c]['price'].values 
           for c in cat_order if c in df_price_cat['category'].values]
cat_labels = [c for c in cat_order if c in df_price_cat['category'].values]

bp = ax.boxplot(bp_data, patch_artist=True, medianprops=dict(color='white', linewidth=1.5))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for element in ['whiskers', 'fliers', 'caps']:
    for item in bp[element]: item.set_color('#8b949e')
    
ax.set_xticks(range(1, len(cat_labels)+1))
ax.set_xticklabels([c[:8] for c in cat_labels], rotation=30, ha='right', color='#8b949e', fontsize=8)
ax.set_title('Price Distribution by Category', color='white', fontsize=12, fontweight='bold')
ax.set_ylabel('Price (₹)', color='#8b949e')
ax.tick_params(colors='#8b949e')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
for spine in ax.spines.values(): spine.set_edgecolor('#30363d')

plt.tight_layout()
plt.savefig('pipeline_analytics.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("📊 Chart saved as pipeline_analytics.png")

In [ ]:
# Success vs Failure donut (if pipeline ran)
total_run = len(successes) + len(failures)
if total_run > 0:
    fig, ax = plt.subplots(figsize=(7, 5))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')
    
    sizes = [len(successes), len(failures)]
    labels = ['Success', 'Failed']
    colors = ['#3fb950', '#f85149']
    
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, autopct='%1.1f%%',
        colors=colors, startangle=90,
        wedgeprops={'edgecolor': '#0d1117', 'linewidth': 3, 'width': 0.6}
    )
    for t in texts: t.set_color('#c9d1d9'); t.set_fontsize(12)
    for at in autotexts: at.set_color('white'); at.set_fontweight('bold'); at.set_fontsize(11)
    
    # Center text
    ax.text(0, 0, f'{len(successes)}\n/{total_run}', 
            ha='center', va='center', color='white', fontsize=18, fontweight='bold')
    
    ax.set_title(f'Pipeline Results — Success Rate: {len(successes)/total_run*100:.1f}%',
                color='white', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('success_rate.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
else:
    print("Run the pipeline with Ollama to see success/failure stats.")

## 11. Summary Statistics

In [ ]:
# Final summary
print("="*55)
print("   LLM PRODUCT NORMALIZER — PIPELINE SUMMARY")
print("="*55)
print(f"  Dataset size      : {len(raw_df):>8,} records")
print(f"  After preprocessing: {len(clean_df):>7,} records")

if settings.output_path.exists():
    df_out = pd.read_csv(settings.output_path)
    df_err = pd.read_csv(settings.error_path) if settings.error_path.exists() else pd.DataFrame()
    total = len(df_out) + len(df_err)
    print(f"  Successful        : {len(df_out):>8,} records")
    print(f"  Errors            : {len(df_err):>8,} records")
    print(f"  Success rate      : {len(df_out)/total*100:>7.1f}%" if total else "  Success rate      : N/A")
    if 'confidence' in df_out.columns:
        print(f"  Avg confidence    : {pd.to_numeric(df_out['confidence'], errors='coerce').mean():>7.3f}")
    if 'price' in df_out.columns:
        prices = pd.to_numeric(df_out['price'], errors='coerce').dropna()
        print(f"  Median price      : ₹{prices.median():>9,.0f}")
elif successes:
    conf_vals = [p.confidence for p in successes]
    print(f"  Successful (sample): {len(successes):>6,} records")
    print(f"  Avg confidence     : {sum(conf_vals)/len(conf_vals):>6.3f}")
else:
    print("  Run 'python -m src.main' to generate full results")

print("="*55)
print()
print("Output files:")
print(f"  📄 {settings.output_path}")
print(f"  📄 {settings.error_path}")
print(f"  📋 {settings.log_path}")

## 12. Conclusion

### What We Built

A **production-grade, fully offline LLM pipeline** that:

| Component | What it does |
|-----------|-------------|
| `preprocess.py` | Cleans raw scraped text (HTML, unicode, whitespace) |
| `llm_parser.py` | Sends prompts to Ollama, retries on failure, extracts JSON |
| `validator.py` | Enforces strict Pydantic schema, routes errors |
| `postprocess.py` | Applies normalization maps for brands/categories |
| `exporter.py` | Writes clean CSV and error CSV |
| `api/app.py` | FastAPI REST API for single + bulk normalization |
| `streamlit/app.py` | Interactive dashboard with upload → normalize → download |

### Key Design Decisions

1. **LLM over regex** — semantic understanding handles arbitrary input variation
2. **Temperature = 0.0** — deterministic, consistent JSON output
3. **Retry + error routing** — resilient pipeline that never crashes
4. **Two-layer quality control** — Pydantic validation + postprocessing maps
5. **Dependency injection** — testable without a live Ollama server

### Next Steps

- Add `asyncio` for parallel processing (5-10x throughput)
- Fine-tune Phi-3-mini on domain-specific data
- Add Great Expectations for data quality monitoring
- Deploy with Kubernetes for horizontal scaling
